# Attacking the contrastive model — the measurement notebooks

How the contrastive models were actually measured: the CL_AT notebook's attack battery, the `attack_analyze2` study (including the unique `pgd_X` variant and a log-KDE attack), and the label-level analyses that ask *which* samples flip and what the adversarial outputs look like.

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `malware_detection_CL_AT.ipynb`, `attack_analyze2.ipynb`, `attack_analyzing.ipynb` and `attack_visualization.ipynb` (originally in `contrastive-learning` and `mal_adv5`). Outputs cleared.*

## Part 1 — The CL_AT battery

### Step 1 — Setup and the SupCon pieces

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py'
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# SupContrast Model (Encoder + Projector)

In [ ]:
import torch
import torch.nn.functional as F

def calculate_divergences(z, labels):
    # Ensure the tensors are on the same device
    device = z.device
    labels = labels.to(device)

    # Compute cosine similarities
    cos_sim_z = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

    # Convert cosine similarities to cosine distances
    cos_distances_z = 1 - cos_sim_z

    # Initialize variables
    N = len(z)
    intra_class_distances = []
    inter_class_distances = []

    for i in range(N):
        # Identify positive and negative sets for the anchor
        positive_indices = (labels == labels[i]).nonzero(as_tuple=True)[0]
        negative_indices = (labels != labels[i]).nonzero(as_tuple=True)[0]

        # Compute intra-class divergence for benign and adversarial examples
        intra_class_distance_z = cos_distances_z[i][positive_indices].mean().item()

        # Compute inter-class divergence for benign and adversarial examples
        inter_class_distance_z = cos_distances_z[i][negative_indices].mean().item()

        intra_class_distances.append(intra_class_distance_z)
        inter_class_distances.append(inter_class_distance_z)

    # Calculate absolute intra-class and inter-class divergences
    d_a_plus = torch.tensor(intra_class_distances).mean().item()
    d_a_minus = torch.tensor(inter_class_distances).mean().item()

    # Calculate relative intra-class divergence (R-DIV)
    d_r_plus = d_a_plus / d_a_minus

    print("Absolute Intra-Class Divergence:", d_a_plus)
    print("Absolute Inter-Class Divergence:", d_a_minus)
    print("Relative Intra-Class Divergence (R-DIV):", d_r_plus)


    return d_a_plus, d_a_minus, d_r_plus

# Example usage
batch_size = 100
latent_dim = 200
num_classes = 2

# Generate random latent representations and labels for demonstration
z = torch.rand(batch_size, latent_dim)
labels = torch.randint(0, num_classes, (batch_size,))

# Calculate divergences
d_a_plus, d_a_minus, d_r_plus = calculate_divergences(z, labels)

In [ ]:
def extract_features(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            output, z = model.CoreModel(inputs.to(torch.float32), return_z=True)
            features.append(z.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels


def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title('t-SNE Scatter Plot')

    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    ax2.set_title('Density Plot')

    plt.show()

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',rounding=False, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4insertion = (gradients >= 0) *(x_var < 1.)*  gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    #if random:
       #round_threshold = torch.rand(x_next.size())
    if rounding:
        x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next


def get_loss(x,y,model):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x)
    loss = criterion(outputs, y.view(-1).long())
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    return loss, done

def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

def PGD_Max2(x,y, model, insertion_array, removal_array, attack_list = ['linf', 'l2', 'l1'],k=50,steps_max=5, rounding=False, is_sample = False, varepsilon = 1e-20, is_report_loss_diff = False):
    """
    PGD_Max adversarial attack.

    Args:
        x: Input data tensor (shape: [samples, features])
        y: Ground truth labels tensor (shape: [samples])
        model: Neural network model
        attack_list: List of norms for attacks (default: ['linf', 'l2', 'l1'])
        steps_max: Maximum number of steps (default: 5)
        is_sample: Flag to sample randomly from the feasible area (default: False)
        vaρεpsilon: Tolerance for stopping condition (default: 1e-20)

    Returns:
        Adversarial version of input data (tensor)
    """
    batch_size = x.shape[0]
    norm_params = {
        'l1': {'k': k, 'step_length': 1.0,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff},
        'l2': {'k': k, 'step_length': 0.5,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff},
        'linf': {'k': k, 'step_length': 0.02,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff}
    }

    model.eval()
    with torch.no_grad():
        loss, done = get_loss(x,y,model) #shape:[samples],[samples]

    pre_loss = loss
    n = x.shape[0]
    adv_x = x.detach().clone().to(x.device)
    stop_flag = torch.zeros(n, dtype=torch.bool).to(x.device) #[samples]

    for t in range(steps_max):
      num_remaining  = (~stop_flag).sum().item()
      #print('number of remaining samples : ',num_remaining )
      if num_remaining  <= 0:
          break

      remaining_label = y[~stop_flag]
      pertbx = []

      for norm in attack_list:
          if norm in norm_params:
              params = norm_params[norm]
              perturbation = pgd_min2(adv_x[~stop_flag], remaining_label, model, insertion_array, removal_array, norm=norm, is_sample=is_sample, **params)
              #print("the number of added features : ", (perturbation.sum() - adv_x[~stop_flag].sum())/len(adv_x[~stop_flag]))
              pertbx.append(perturbation)
          else:
              raise ValueError("Expected 'l1' or 'l2' or 'linf' norm.")


      # here pertbx.shape = a list of (number of attacks  ,(num_remaining ,features))
      pertbx = torch.vstack(pertbx)
      # here pertbx.shape = a tensor (num_remaining *number of attacks samples, features)

      with torch.no_grad():
        remaining_label_ext = torch.cat([remaining_label] * len(attack_list)) #(labels*number of attacks )
        loss, done = get_loss(pertbx, remaining_label_ext,model) #(labels*number of attacks )

        # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
        # while if no attacks evade the victim successful, all perturbed examples are reminded for selection
        max_v = loss.amax()
        loss[done] += max_v

        loss = loss.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)
        done = done.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)

        success_flag = torch.any(done, dim=-1) #(num_remaining )

        pertbx = pertbx.reshape(len(attack_list), num_remaining , x.shape[1]).permute([1, 0, 2])#(num_remaining ,attacks,features)
        _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
        adv_x[~stop_flag] = pertbx[torch.arange(num_remaining ), indices]
        a_loss = loss[torch.arange(num_remaining ), indices]
        pre_stop_flag = stop_flag.clone()
        stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
        pre_loss[~pre_stop_flag] = a_loss

    return adv_x

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
import torch
import torch.nn.functional as F

def distance(x, y, dist, pairwise=False):
    """
        Pairwise distance
        Args:
            x, y: input pair
            dist: distance type ["l2", "l1", "linf", "cosine"]
            pairwise:
        Note:
            Need renormalize if using l1
    """

    assert(len(x.shape) == 2)
    b, d = x.shape

    if pairwise:
        x = torch.unsqueeze(x, dim=0) # [1, b, d]
        y = torch.unsqueeze(y, dim=1) # [b, 1, d]

    if dist == 'l2':
        loss = torch.sqrt(torch.sum(torch.square(x - y), dim=-1, keepdim=False)) # [b,b]

    elif dist == 'l1':
        loss = torch.sum(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'linf':
        loss, _ = torch.max(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'lkinf':
        k = 10
        t, _ = torch.topk(torch.abs(x - y), k=k, dim=-1)
        loss = torch.sum(t, dim=-1, keepdim=False)
        loss /= k

    elif dist == 'cosine':
        # Note that: cosine_similarity range [-1,1]
        # dist=-1 <--> very similar
        # dist=0 <--> orthogonal
        # dist=1 <--> very disimilar
        loss = 0 - torch.cosine_similarity(x, y, dim=-1)

    elif dist == 'matmul':
        x = torch.squeeze(x) # [b,d]
        y = torch.squeeze(y) # [b,d]
        loss = - torch.matmul(x, y.T)

    # loss = loss / torch.max(loss).detach() # normalizing to avoid numerical problem

    if pairwise:
        assert(loss.shape[0] == b)
        assert(loss.shape[1] == b)
    else:
        assert(loss.shape[0] == x.shape[0]) # [b, ]

    return loss

def get_positive_mask(labels):
    # ATTENTION HERE, positive mask will ignore the diagonal

    # l = torch.matmul(labels, labels.T) # [2b,2b]
    # m = torch.ones_like(l).fill_diagonal_(0) # [2b,2b]
    # pos_mask = torch.multiply(l, m) # [2b,2b]

    l = torch.matmul(labels, labels.T).fill_diagonal_(0) # [2b,2b]
    return l

def get_negative_mask(labels):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    return inv_mask

def get_hard_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    hard_mask = torch.matmul(labels, preds.T) # [2b, 2b], mask for tile(y_j) == y_i
    neg_mask = torch.multiply(inv_mask, hard_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask


def get_soft_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    soft_mask = torch.matmul(preds, preds.T) # [2b, 2b], mask for tile(y_j) == tile(y_i)
    neg_mask = torch.multiply(inv_mask, soft_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask

LARGE_NUM = 1e9

def soft_lcscl(hidden,
             labels,
             preds,
             dist,
             neg_type,
             hidden_norm=True,
             temperature=0.07):
    """Compute Local Supervised Contrastive Loss.
    Args:
        hidden: hidden vector (`Tensor`) of shape (2 * bsz, dim).
        labels: labels of hidden vector of shape (2*bsz, classes)
        preds: current prediction
        hidden_norm: whether or not to use normalization on the hidden vector.
        temperature: a `floating` number for temperature scaling.
        dist: distance metric in latent space {'l1', 'l2', 'cosine', 'linf'}
        neg_type:
            'hard': negative set when y_j ~= y_i, but tilde(y_j) == y_i
            'soft': negative set when y_j ~= y_i, but tilde(y_j) == tilde(y_i)
            'merge': negative set when y_i ~= y_i but tilde(y_j) in [y_i,  tilde(y_i)]
            'leaking':
    Returns:
        A loss scalar.

    the choice of temperature:
        temperature = 0.5 based on simCLR paper
        temperature = 0.07 based on SCL paper (they claim that smaller temperature
        benefit training more than higher ones)
    """

    # Get (normalized) hidden1 and hidden2.
    if hidden_norm:
        # gradient will pass through the normalization step
        hidden = F.normalize(hidden, p=2, dim=-1)


    batch_size = hidden.shape[0]

    # Remove illegal elements
    same_preds = get_positive_mask(preds)
    same_labels = get_positive_mask(labels)
    diff_labels = get_negative_mask(labels)

    same_preds_same_labels = torch.multiply(same_preds, same_labels)
    same_preds_diff_labels = torch.multiply(same_preds, diff_labels)

    pos_mask = get_positive_mask(labels)
    # neg_mask = tf.ones_like(pos_mask) - pos_mask # INCORRECT, because it consider the diagonal as NEGATIVE
    neg_mask = get_negative_mask(labels) # CORRECT
    hard_neg_mask = get_hard_negative_mask(labels, preds)
    soft_neg_mask = get_soft_negative_mask(labels, preds)
    merge_mask = torch.minimum(hard_neg_mask+soft_neg_mask, torch.ones_like(hard_neg_mask))

    # ATTENTION HERE, NEED VERIFIED BY EXPERIMENT, Stop_gradient make it worse
    # pos_mask = pos_mask.detach()
    # neg_mask = neg_mask.detach()
    # hard_neg_mask = hard_neg_mask.detach()
    # soft_neg_mask = soft_neg_mask.detach()
    # merge_mask = merge_mask.detach()
    # same_preds_same_labels = same_preds_same_labels.detach()
    #
    if neg_type == 'all':
        illegal_mask = torch.zeros_like(neg_mask)
        nb_neg = torch.sum(neg_mask, dim=-1)

    elif neg_type == 'hard':
        illegal_mask = torch.maximum(neg_mask - hard_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(hard_neg_mask, dim=-1)

    elif neg_type == 'soft':
        illegal_mask = torch.maximum(neg_mask - soft_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(soft_neg_mask, dim=-1)

    elif neg_type == 'merge':
        illegal_mask = torch.maximum(neg_mask - merge_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(merge_mask, dim=-1)

    elif neg_type == 'leaking':
        illegal_mask = torch.maximum(torch.ones_like(same_preds_same_labels) - same_preds_same_labels - same_preds_diff_labels, torch.zeros_like(same_preds_same_labels))
        #print(illegal_mask)
        nb_neg = torch.sum(same_preds_diff_labels, dim=-1)

    # Pairwise Similarity
    masks = F.one_hot(torch.arange(batch_size), num_classes=batch_size).to(hidden.device)
    sim = - distance(hidden, hidden, dist, pairwise=True) / temperature
    logits_12 = sim - masks * LARGE_NUM
    logits_masked = logits_12 - LARGE_NUM * illegal_mask
    logprobs = F.log_softmax(logits_masked, dim=1)

    """ Convert labels to pairwise labels of shape [2b, 2b], diagonal(labels)=0
        multiple hot label vectors

    """
    # Cross entropy loss with multiple hot labels

    if neg_type == 'leaking':
        loss = - torch.sum(same_preds_same_labels * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(same_preds_same_labels, dim=-1), torch.ones_like(loss)) # [2b, ]
    else:
        loss = - torch.sum(pos_mask * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(pos_mask, dim=-1), torch.ones_like(loss)) # [2b, ]

    # if True:
    #     scale = torch.div(nb_neg, nb_neg+nb_pos) # V1
    #     # scale = torch.div(nb_neg, nb_pos) #V3
    #     # scale = nb_neg #V2
    #     loss = torch.multiply(loss, scale)

    assert(loss.shape == nb_pos.shape)
    #print(nb_pos)
    loss = torch.div(loss, nb_pos) # [2b,]
    loss = torch.mean(loss, dim=0) # [1,]

    return loss

### Step 2 — Training, then the attack battery

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

def adversarial_training(model, train_loader, val_loader, attack, epochs=100, lr=0.001, weight_decay=0., beta=1., device='cuda', verbose=True, **kwargs):

    # Define parameters for the loss function
    dist = 'matmul'
    neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
    hidden_norm = False
    temperature = 0.07


    # Define Loss Function and Optimizer
    #class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion_at = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.1, 0.999), weight_decay=weight_decay)
    momentum = 5e-4
    #optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)


    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(epochs):
        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0
        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])

            # bsz must be updated due to added adversarial samples
            bsz = x_batch.shape[0]

            model.train()
            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            #print(f'Training loss (batch level): {loss_train.item():.4f} | Train accuracy: {((predicted == y_batch.view(-1)).sum().item()/bsz):.2f}%')

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz


        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        #if (pre_loss - mean_loss) < 1e-2:
            #target_model.load_state_dict(model.state_dict())
            # because after new adversarial samples loss increase so mostly after just one epoch this condition would be true
            # so we change pre_loss to inf for training more than one epoch
            #mean_loss = torch.inf


        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        if verbose:
            print(f"\tVal accuracy (without attack) {(avg_acc_val) * 100:.4}% and accuracy (with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel selected at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

def adversarial_training2(model, train_loader, val_loader, attack, epochs=100, lr=0.001, weight_decay=0., beta=1., device='cuda', verbose=True, **kwargs):

    # Initialize target model
    target_model = deepcopy(model)
    target_model.eval()

    # Define parameters for the loss function
    dist = 'matmul'
    neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
    hidden_norm = False
    temperature = 0.07


    # Define Loss Function and Optimizer
    #class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion_at = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.1, 0.999), weight_decay=weight_decay)
    momentum = 5e-4
    #optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)


    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0
    mean_loss = torch.inf
    i = 1
    for epoch in range(epochs):
        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0
        pre_loss = mean_loss
        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, target_model, **kwargs)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])

            # bsz must be updated due to added adversarial samples
            bsz = x_batch.shape[0]

            model.train()
            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            #print(f'Training loss (batch level): {loss_train.item():.4f} | Train accuracy: {((predicted == y_batch.view(-1)).sum().item()/bsz):.2f}%')

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz

            if i % 4 == 0:
              target_model.load_state_dict(model.state_dict())

            i+=1

        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        #if (pre_loss - mean_loss) < 1e-2:
            #target_model.load_state_dict(model.state_dict())
            # because after new adversarial samples loss increase so mostly after just one epoch this condition would be true
            # so we change pre_loss to inf for training more than one epoch
            #mean_loss = torch.inf


        target_model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = target_model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, target_model, **kwargs)
            outputs = target_model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(target_model.state_dict(), 'model_AT_max.pth')

        if verbose:
            print(f"\tVal accuracy (without attack) {(avg_acc_val) * 100:.4}% and accuracy (with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel selected at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 1024
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

epochs = 400

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':10, 'rounding':False,'steps_max': 2, 'is_report_loss_diff' : False}
#attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'norm':'l2','k':20, 'rounding':False,'step_length': 0.5, 'is_report_loss_diff' : False}
#pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',rounding=False, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
adversarial_training(combined_model.to(device), train_loader, val_loader, PGD_Max2, epochs, lr=0.005, weight_decay=1e-4, beta=1., device=device, verbose=True, **attack_params)

In [ ]:
combined_model2 = Wrapper(core_model, proj_head).to(device)
combined_model2.load_state_dict(torch.load('/content/model_AT_max.pth'))

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':100,'steps_max': 5, 'rounding':True, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2.to(device), PGD_Max2, device, **attack_params)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":100, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":100, 'is_report_loss_diff' : False}),
    (bca_k, {"k":100, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":100, 'is_report_loss_diff' : False}),
    #(mimicry, {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':1,'is_sample':False, 'varepsilon':1e-20}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':5,'is_sample':False, 'varepsilon':1e-20}),
    (StepwiseMax_onestep2, {'insertion_array':insertion_array, 'removal_array':removal_array,'step_lengths':{"l1": 1.0, "l2": 0.05, "linf": 0.0008}, "steps":650}),

]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model2.to(device), attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

In [ ]:
features, labels = extract_features(combined_model2, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

### Step 3 — Mimicry, evaluation, and the second training cycle

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":30, 'is_report_loss_diff':False}
adv_predict(test_loader, combined_model2.to(device), mimicry, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}
adv_predict(test_loader, combined_model2.to(device), mimicry, device, **attack_params)

In [ ]:
model_evaluation(combined_model2.to(device), test_loader, device)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 128
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

epochs = 100

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':10, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}

adversarial_training4(combined_model.to(device), train_loader, val_loader, PGD_Max2, adv_epochs=100, lr=0.002, weight_decay=0., beta=1., device=device, verbose=True, **attack_params)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":100, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":100, 'is_report_loss_diff' : False}),
    (bca_k, {"k":100, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":100, 'is_report_loss_diff' : False}),
    #(mimicry, {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':1,'is_sample':False, 'varepsilon':1e-20}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':5,'is_sample':False, 'varepsilon':1e-20}),
    (StepwiseMax_onestep2, {'insertion_array':insertion_array, 'removal_array':removal_array,'step_lengths':{"l1": 1.0, "l2": 0.05, "linf": 0.0008}, "steps":650}),

]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model2, attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

## Part 2 — attack_analyze2

### Step 4 — The models, loaded and evaluated (incl. combined_model_adam)

# Loading data

In [ ]:
#RBF_models

download_links = [
                  'https://drive.google.com/uc?id=1-8lJXLdAl_4NdDwzw9kFML0aiOCTrI9f',
                  'https://drive.google.com/uc?id=1-OHACrNCt0yKBbdqQPVfNZcjKt5_jxKD',
                  'https://drive.google.com/uc?id=1-KeXJXtU1_6m9JOhormeVwigy0myX3HL',
                  'https://drive.google.com/uc?id=1-13RDdZqnrNkdHg3D8PC5KI0CZREwlsz',
                  'https://drive.google.com/uc?id=1-8LjsCdzKh6asxCFsYLiQZbSEXXKSQBP',

]

import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM_weightedLoss.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/combined_model_adam.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/combined_model_adam_state_dict.pth'
]

In [ ]:
import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from adverserial_attacks_functions import *


torch.manual_seed(0)

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    #os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
# Define the Encoder Network
class Encoder(nn.Module):
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        #self.dropout = nn.Dropout(dropout_prob)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        #x = self.dropout(x)
        x = self.fc2(x)
        out = self.relu(x)
        return out

# Define the SupConNet Network
class SupConNet(nn.Module):
    """Backbone + projection head"""
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6, feat_dim=128):
        super(SupConNet, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim1, hidden_dim2, dropout_prob)
        self.head = nn.Linear(hidden_dim2, feat_dim)

    def forward(self, x):
        feat = self.encoder(x)
        feat = F.normalize(self.head(feat), dim=1)
        return feat

# Define the Linear Classifier
class LinearClassifier(nn.Module):
    """Linear classifier"""
    def __init__(self, feat_dim=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, features):
        return self.fc(features)

class CombinedModel(nn.Module):
    def __init__(self, encoder, classifier):
        super(CombinedModel, self).__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, x):
        features = self.encoder(x)
        output = self.classifier(features)
        return output

In [ ]:
model_DNN = MalwareDetectionModel().to(device)
# Load model parameters
model_DNN.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM_weightedLoss = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM_weightedLoss.load_state_dict(torch.load('model_AT_rFGSM_weightedLoss.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
encoder = Encoder().to(device)
classifier = LinearClassifier().to(device)
combined_model = CombinedModel(encoder, classifier).to(device)

# Load model parameters
combined_model.load_state_dict(torch.load('combined_model_adam_state_dict.pth', map_location=torch.device(device)))

In [ ]:
combined_model = torch.load('combined_model_adam.pth', map_location=torch.device(device))
# Now, set requires_grad=True for all parameters
for param in combined_model.parameters():
    param.requires_grad = True

In [ ]:
model_evaluation(combined_model, test_loader, device)

In [ ]:
model_evaluation(model_AT_imax, test_loader, device)

In [ ]:
model_evaluation(model_DNN, test_loader, device)

In [ ]:
model_evaluation(model_AT_rFGSM,test_loader,device)

In [ ]:
model_evaluation(model_AT_rFGSM_weightedLoss,test_loader,device)

In [ ]:
for x,y in test_loader:
  print(x.shape)
  print(y.shape)
  break

bens = x[y.squeeze()==0]
bens_y = y[y.squeeze()==0]
print(bens.shape)

mals = x[y.squeeze()==1]
mals_y = y[y.squeeze()==1]
print(mals.shape)

### Step 5 — pgd_min2, the pgd_X variant, and pgd2 — runs and losses

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

In [ ]:
def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_d

In [ ]:
def pgd_X(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    model_output = model(x)
    Z_y = model_output[torch.arange(model_output.size(0)), y.view(-1).long()]
    Z_t = model_output[torch.arange(model_output.size(0)), ~y.view(-1).long()]
    loss_natural = Z_y - Z_t

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        model_output = model(x_var)
        Z_y = model_output[torch.arange(model_output.size(0)), y.view(-1).long()]
        Z_t = model_output[torch.arange(model_output.size(0)), ~y.view(-1).long()]
        loss = Z_y - Z_t
        #loss = (Z_y - Z_t)/(torch.max(Z_y , Z_t) - torch.min(Z_y , Z_t))

        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        #loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())

        loss2 = criterion(model_output, torch.zeros_like(y.view(-1).long()))
        loss_steps_d.append(loss2.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    model_output = model(x_next)
    Z_y = model_output[torch.arange(model_output.size(0)), y.view(-1).long()]
    Z_t = model_output[torch.arange(model_output.size(0)), ~y.view(-1).long()]
    loss_adv = (Z_y - Z_t).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_d

In [ ]:
def pgd2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack.
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), y.view(-1).long())

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, y.view(-1).long())
        loss2 = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_d.append(loss2.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = grad_vars[0].data

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), y.view(-1).long()).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv < loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        outputs = model(x_next)
        _, predicted = torch.topk(outputs, k=1)
        done = (predicted != y).squeeze()
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next,loss_steps_d

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    #outputs = model(x_test)
    #predicted = outputs.argmax(1).unsqueeze(1)
    #acc_test = (predicted == y_test).sum().item() / len(y_test)
    #avg_acc_test.append(acc_test)

    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    break

In [ ]:
x_next,loss_steps_d1 = pgd2(mal_x_batch, mal_y_batch, model_AT_imax, insertion_array, removal_array, k=250, step_length=0.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next,loss_steps_d2 = pgd_min2(mal_x_batch, mal_y_batch, model_AT_imax, insertion_array, removal_array, k=250, step_length=0.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)
x_next,loss_steps_d3 = pgd_X(mal_x_batch, mal_y_batch, model_AT_imax, insertion_array, removal_array, k=250, step_length=0.01, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False)




# Create the plot
plt.figure(figsize=(10, 6))

# Plot each list
plt.plot(loss_steps_d1, label='pgd2', marker='o')
plt.plot(loss_steps_d2, label='pgd_min2', marker='s')
plt.plot(loss_steps_d3, label='pgd_X', marker='^')

# Add labels and title
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Multiple Lines in One Plot')

# Add a legend
plt.legend()

# Show the plot
plt.show()

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x,_= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : True}
adv_predict(test_loader, model_AT_imax, pgd2, device, **attack_params)

In [ ]:
def get_loss(x,y,model):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x)
    loss = criterion(outputs, y.view(-1).long())
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    return loss, done

def PGD_Max2(x,y, model, insertion_array, removal_array,attack_approach, attack_list = ['linf', 'l2', 'l1'],steps_max=5, is_sample = False, varepsilon = 1e-20, is_report_loss_diff = False):
    """
    PGD_Max adversarial attack.

    Args:
        x: Input data tensor (shape: [samples, features])
        y: Ground truth labels tensor (shape: [samples])
        model: Neural network model
        attack_list: List of norms for attacks (default: ['linf', 'l2', 'l1'])
        steps_max: Maximum number of steps (default: 5)
        is_sample: Flag to sample randomly from the feasible area (default: False)
        vaρεpsilon: Tolerance for stopping condition (default: 1e-20)

    Returns:
        Adversarial version of input data (tensor)
    """
    batch_size = x.shape[0]
    norm_params = {
        'l1': {'k': 50, 'step_length': 1.0, 'is_report_loss_diff':is_report_loss_diff},
        'l2': {'k': 50, 'step_length': 0.5, 'is_report_loss_diff':is_report_loss_diff},
        'linf': {'k': 50, 'step_length': 0.02, 'is_report_loss_diff':is_report_loss_diff}
    }

    model.eval()
    with torch.no_grad():
        loss, done = get_loss(x,y,model) #shape:[samples],[samples]

    pre_loss = loss
    n = x.shape[0]
    adv_x = x.detach().clone().to(x.device)
    stop_flag = torch.zeros(n, dtype=torch.bool).to(x.device) #[samples]

    for t in range(steps_max):
      num_remaining  = (~stop_flag).sum().item()
      #print('number of remaining samples : ',num_remaining )
      if num_remaining  <= 0:
          break

      remaining_label = y[~stop_flag]
      pertbx = []

      for norm in attack_list:
          if norm in norm_params:
              params = norm_params[norm]
              perturbation,_ = attack_approach(adv_x[~stop_flag], remaining_label, model, insertion_array, removal_array, norm=norm, is_sample=is_sample, **params)
              #print("the number of added features : ", (perturbation.sum() - adv_x[~stop_flag].sum())/len(adv_x[~stop_flag]))
              pertbx.append(perturbation)
          else:
              raise ValueError("Expected 'l1' or 'l2' or 'linf' norm.")


      # here pertbx.shape = a list of (number of attacks  ,(num_remaining ,features))
      pertbx = torch.vstack(pertbx)
      # here pertbx.shape = a tensor (num_remaining *number of attacks samples, features)

      with torch.no_grad():
        remaining_label_ext = torch.cat([remaining_label] * len(attack_list)) #(labels*number of attacks )
        loss, done = get_loss(pertbx, remaining_label_ext,model) #(labels*number of attacks )

        # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
        # while if no attacks evade the victim successful, all perturbed examples are reminded for selection
        max_v = loss.amax()
        loss[done] += max_v

        loss = loss.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)
        done = done.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)

        success_flag = torch.any(done, dim=-1) #(num_remaining )

        pertbx = pertbx.reshape(len(attack_list), num_remaining , x.shape[1]).permute([1, 0, 2])#(num_remaining ,attacks,features)
        _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
        adv_x[~stop_flag] = pertbx[torch.arange(num_remaining ), indices]
        a_loss = loss[torch.arange(num_remaining ), indices]
        pre_stop_flag = stop_flag.clone()
        stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
        pre_loss[~pre_stop_flag] = a_loss

    return adv_x,adv_x

In [ ]:
#pgd_min2
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_approach':pgd_min2,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_imax, PGD_Max2, device, **attack_params)

In [ ]:
#pgd_X
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_approach':pgd_X,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_imax, PGD_Max2, device, **attack_params)

In [ ]:
#pgd_min2
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_approach':pgd2,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_imax, PGD_Max2, device, **attack_params)

### Step 6 — Mimicry at growing trial counts, and the KDE-log attack

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    if (attack ==  gkde) or (attack ==  mimicry):
      # Pre-select benign samples
      benign_samples = []
      for x_batch, y_batch in test_loader:
        benign_samples.append(x_batch[y_batch.squeeze() == 0])

      ben_x = torch.cat(benign_samples, dim=0).to(device)

      # Select the top 500 high confidence benign samples
      top_500_high_confidence_benign_samples = ben_x[:500]



    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            if attack == mimicry:
                pertb_mal_x = mimicry(top_500_high_confidence_benign_samples, mal_x_batch, model, **kwargs)
            elif attack == gkde:
                with torch.enable_grad():
                  pertb_mal_x = gkde(mal_x_batch, mal_y_batch, model, top_500_high_confidence_benign_samples, **kwargs)
            else :
                with torch.enable_grad():
                    pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":10, 'is_report_loss_diff':False}
adv_predict(test_loader, model_AT_imax, mimicry, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":200, 'is_report_loss_diff':False}
adv_predict(test_loader, model_AT_imax, mimicry, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":1000, 'is_report_loss_diff':False}
adv_predict(test_loader, model_AT_imax, mimicry, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":2000, 'is_report_loss_diff':False}
adv_predict(test_loader, model_AT_imax, mimicry, device, **attack_params)

In [ ]:
def KDE(x, data, bandwidth, dissimilarity='euclidean', kernel='gaussian'):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.
        dissimilarity (str): The type of dissimilarity to use ('cosine' or 'euclidean').
        kernel (str): The type of kernel to use ('gaussian' is supported).

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate dissimilarity for all data points
    if dissimilarity == 'cosine':
        u = cosine_dissimilarity(x, data)
    elif dissimilarity == 'euclidean':
        u = torch.abs(x[:, None, :] - data)

    # Compute kernel contributions for all data points
    if kernel == 'gaussian':
        if dissimilarity == 'cosine':
            kernel_contributions = torch.exp(-u**2 / bandwidth_tensor**2)
        elif dissimilarity == 'euclidean':
            kernel_contributions = torch.exp(-torch.sum(u**2, dim=-1) / bandwidth_tensor**2)
    else:
        if dissimilarity == 'cosine':
            kernel_contributions = torch.exp(-u / bandwidth_tensor)
        elif dissimilarity == 'euclidean':
            kernel_contributions = torch.exp(-torch.sum(u, dim=-1) / bandwidth_tensor)

    # Sum contributions across all data points
    estimate = torch.mean(kernel_contributions, dim=1)

    return estimate

In [ ]:
def pgd_min2_kde_log(x, y, model,centers, bandwidth, dissimilarity, penalty_factor, kernel, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',
                 initial_rounding_threshold=0.5, round_threshold=0.5, decaying_penalty=False, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    eps = 1e-10
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long())) +  penalty_factor * KDE(x, centers, bandwidth, dissimilarity,kernel)

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_d = []
    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        ce = criterion(model(x_var), torch.zeros_like(y.view(-1).long()))
        k = KDE(x_var, centers, bandwidth, dissimilarity,kernel)
        kde = -torch.log(k + eps)

        if decaying_penalty == True:
            loss = ce + (1./(1.2)**t) * penalty_factor * kde
        else:
            loss = ce +  penalty_factor * kde



        #loss_steps.append(loss.mean().detach().item())
        loss_steps_d.append(criterion(model(x_var), y.view(-1).long()).mean().detach().item())
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
# Pre-select benign samples
benign_samples = []
for x_batch, y_batch in test_loader:
  benign_samples.append(x_batch[y_batch.squeeze() == 0])

ben_x = torch.cat(benign_samples, dim=0).to(device)

# Select the top 500 high confidence benign samples
centers = ben_x[:500]

In [ ]:
attack_params =  {'centers':centers,'bandwidth': 1,'dissimilarity':'euclidean','penalty_factor': 1,'kernel':'laplac','insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 0.5, 'norm': 'l2', 'decaying_penalty':False, 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_imax, pgd_min2_kde_log, device, **attack_params)

## Part 3 — Label-level analysis

### Step 7 — Which samples flip? (from `attack_analyzing.ipynb`)

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.6)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

    def forward_from_z(self, z):
        output = self.fc3(z)
        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

combined_model = Wrapper(core_model, proj_head).to(device)
combined_model.load_state_dict(torch.load('/content/model_AT_max (4).pth', map_location=torch.device(device)))

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 4096
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()
            break

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return ben_x_batch,mal_x_batch,pertb_mal_x

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':10, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}
ben_x_batch,mal_x_batch,pertb_mal_x = adv_predict(test_loader, combined_model, PGD_Max2, device, **attack_params)

outputs, benign_samples = combined_model.CoreModel(ben_x_batch.to(torch.float32), return_z=True)
outputs, malicious_samples = combined_model.CoreModel(mal_x_batch.to(torch.float32), return_z=True)
outputs, adversarial_samples = combined_model.CoreModel(pertb_mal_x.to(torch.float32), return_z=True)

In [ ]:
outputs = combined_model(pertb_mal_x)
adversarial_labels = outputs.argmax(1)

In [ ]:
outputs = combined_model(mal_x_batch)
malicious_labels = outputs.argmax(1)

In [ ]:
outputs = combined_model(pertb_mal_x[(malicious_labels == 1) & (adversarial_labels == 0)])
probabilities = F.softmax(outputs, dim=1)

print(probabilities)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# Assuming benign_samples, malicious_samples, adversarial_samples, malicious_labels, and adversarial_labels are your input tensors/arrays
# Convert them to numpy arrays if they are not already
benign_samples = np.array(benign_samples.detach())
malicious_samples = np.array(malicious_samples.detach())
adversarial_samples = np.array(adversarial_samples.detach())
malicious_labels = np.array(malicious_labels.detach())  # Model output for malicious samples (0: wrong, 1: correct)
adversarial_labels = np.array(adversarial_labels.detach())  # Model output for adversarial samples (0: wrong, 1: correct)

# Combine all samples into one array for t-SNE processing
all_samples = np.concatenate((benign_samples, malicious_samples, adversarial_samples), axis=0)

# Create labels to distinguish the samples
labels = np.concatenate((
    np.zeros(len(benign_samples)),  # Label 0 for benign samples
    np.ones(len(malicious_samples)),  # Label 1 for malicious samples
    np.full(len(adversarial_samples), 2)  # Label 2 for adversarial examples
))

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(all_samples)

# Separate the transformed results
benign_tsne = tsne_results[labels == 0]
malicious_tsne = tsne_results[labels == 1]
adversarial_tsne = tsne_results[labels == 2]

# Categorize adversarial examples based on the criteria
adv_label_0 = adversarial_tsne[(malicious_labels == 0)]  # Malicious samples detected wrongly by model
adv_label_1 = adversarial_tsne[(malicious_labels == 1) & (adversarial_labels == 1)]  # Malicious detected correctly, adversarial detected correctly
adv_label_2 = adversarial_tsne[(malicious_labels == 1) & (adversarial_labels == 0)]  # Malicious detected correctly, adversarial detected wrongly

# Plot the t-SNE results using seaborn
plt.figure(figsize=(36, 24))
sns.scatterplot(x=benign_tsne[:, 0], y=benign_tsne[:, 1], label='Benign Samples', alpha=0.6, s=100)
sns.scatterplot(x=malicious_tsne[:, 0], y=malicious_tsne[:, 1], label='Malicious Samples', alpha=0.6, s=100)
sns.scatterplot(x=adv_label_0[:, 0], y=adv_label_0[:, 1], label='Adv. Examples (Malicious Wrong)', alpha=0.6, s=100)
sns.scatterplot(x=adv_label_1[:, 0], y=adv_label_1[:, 1], label='Adv. Examples (Malicious Correct, Adv. Correct)', alpha=0.6, s=100)
sns.scatterplot(x=adv_label_2[:, 0], y=adv_label_2[:, 1], label='Adv. Examples (Malicious Correct, Adv. Wrong)', alpha=0.6, s=100)

# Draw lines connecting malicious samples to their corresponding adversarial examples
for i in range(len(malicious_samples)):
    plt.plot([malicious_tsne[i, 0], adversarial_tsne[i, 0]], [malicious_tsne[i, 1], adversarial_tsne[i, 1]], 'k--')

plt.legend()
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('t-SNE Plot of Benign, Malicious, and Adversarial Samples')
plt.show()

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':10, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}
ben_x_batch,mal_x_batch,pertb_mal_x = adv_predict(test_loader, combined_model, PGD_Max2, device, **attack_params)

In [ ]:
PGD_Max2(mal_x_batch, mal_y_batch, model, **attack_params)

In [ ]:
model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        if verbose:
            print(f"\tVal accuracy (without attack) {(avg_acc_val) * 100:.4}% and accuracy (with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel selected at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
import copy

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':10, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}

# Deep copy the model
model = copy.deepcopy(combined_model)

model.eval()
cor_val = 0
cor_ad_val = 0
n_samples = 0
n_ad_samples = 0
outputs = []
ad_outputs = []
labels = []
ad_labels = []
for x_val, y_val in val_loader:
    x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
    n_samples += len(x_val)
    output = model(x_val)
    outputs.append(output)
    labels.append(y_val)

    mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
    n_ad_samples += len(mal_x_batch)
    pertb_mal_x = PGD_Max2(mal_x_batch, mal_y_batch, model, **attack_params)
    ad_output = model(pertb_mal_x)
    ad_outputs.append(ad_output)
    ad_labels.append(mal_y_batch)

# Concatenate along the first dimension (rows)
outputs_tensor = torch.cat(outputs, dim=0)
ad_outputs_tensor = torch.cat(ad_outputs, dim=0)
labels_tensor = torch.cat(labels, dim=0)
ad_labels_tensor = torch.cat(ad_labels, dim=0)


print(outputs_tensor.shape)
print(ad_outputs_tensor.shape)
print(labels_tensor.shape)
print(ad_labels_tensor.shape)

In [ ]:
# just normal samples

true_labels = labels_tensor

# Apply the softmax function along the second dimension (columns)
probabilities = F.softmax(outputs_tensor, dim=1)

# Get the probabilities of being malware (second column)
malware_probabilities = probabilities[:, 1].detach().numpy()

# Define different thresholds for classification
thresholds = [0.1,0.15,0.2,0.25,0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
results = []

for threshold in thresholds:
    predictions = (malware_probabilities > threshold).astype(int)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)
    results.append((threshold, accuracy, precision, recall, f1))

for result in results:
    print(f"Threshold: {result[0]}, Accuracy: {result[1]}, Precision: {result[2]}, Recall: {result[3]}, F1-Score: {result[4]}")

# Plot the ROC Curve
fpr, tpr, thresholds = roc_curve(true_labels, malware_probabilities)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:0.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

In [ ]:
# normal samples + adversarial samples

outputs = torch.cat((outputs_tensor, ad_outputs_tensor), dim=0)
true_labels = torch.cat((labels_tensor, ad_labels_tensor), dim=0)

# Apply the softmax function along the second dimension (columns)
probabilities = F.softmax(outputs, dim=1)

# Get the probabilities of being malware (second column)
malware_probabilities = probabilities[:, 1].detach().numpy()

# Define different thresholds for classification
thresholds = [0.1,0.15,0.2,0.25,0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
results = []

for threshold in thresholds:
    predictions = (malware_probabilities > threshold).astype(int)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)
    results.append((threshold, accuracy, precision, recall, f1))

for result in results:
    print(f"Threshold: {result[0]}, Accuracy: {result[1]}, Precision: {result[2]}, Recall: {result[3]}, F1-Score: {result[4]}")

# Plot the ROC Curve
fpr, tpr, thresholds = roc_curve(true_labels, malware_probabilities)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:0.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

In [ ]:
true_labels = ad_labels_tensor

# Apply the softmax function along the second dimension (columns)
probabilities = F.softmax(ad_outputs_tensor, dim=1)

# Get the probabilities of being malware (second column)
malware_probabilities = probabilities[:, 1].detach().numpy()

# Define different thresholds for classification
thresholds = [0.1,0.15,0.2,0.25,0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
results = []

for threshold in thresholds:
    predictions = (malware_probabilities > threshold).astype(int)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)
    results.append((threshold, accuracy, precision, recall, f1))

for result in results:
    print(f"Threshold: {result[0]}, Accuracy: {result[1]}")

### Step 8 — Visualising the adversarial outputs (from `attack_visualization.ipynb`)

In [ ]:
z = torch.randn(10, 200)  # Example z tensor
lables = torch.zeros(10,dtype=int)
output = combined_model.CoreModel.forward_from_z(z)
output

In [ ]:
criterion = nn.CrossEntropyLoss(reduction='none')
criterion(output, lables)

In [ ]:
outputs = combined_model(pertb_mal_x)
y_pred = outputs.argmax(1)
y_pred